In [ ]:
print('hello')

hello


# Load dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")  # for example
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


# Load a tokenizer and tokenize the text

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    # truncation=True: cuts long sentences to 'max_length'
    # padding="max_length": ensures all sequences are the same langth (needed for batching)

tokenized_dataset = dataset.map(preprocess_function, batched=True) # batched=True: processes multiple examples at once -> faster

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

# Load a model for my task

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2) # num_labels=2: matches my task (binary classification/sentiment=2; positive vs negative)
# The same model architecture could also be used for multi-class (just change num_labels):
# num_labels=3: 3-way classification; sentiment e.g., negative, neutral, positive
# Therefore, num_labels = number of distinct classes in your dataset!

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


num_labels = number of distinct classes in your dataset

# Train and evaluate using the Trainer API

In [ ]:
from transformers import TrainingArguments, Trainer
from evaluate import load

accuracy = load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments( # defines training config.
    output_dir="results",
    evaluation_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].shuffle(seed=42).select(range(1000)),  # small subset
    eval_dataset=tokenized_dataset["test"].select(range(200)),
    compute_metrics=compute_metrics,
)

trainer.train()

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'